# 第 1 周末练习 —— 技术问答解释器

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、精炼的解释（定义 / 原理 / 示例按需组合）
- **额外要求**：用**流式（streaming）**一边生成一边更新 Markdown，而不是等整段答完才一次性显示

这是你在课程期间自己也能天天用的工具；第 2 周之后还可以再加用户界面。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | `system_prompt` 定讲解结构，`question` 放具体问题 |
| 流式输出 `stream=True` | 逐块拼接 + `update_display` 刷新 Markdown |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2:3b`（常量 `MODEL_LLAMA`），经 OpenAI 兼容 `/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env` 中的 `OPENAI_API_KEY`；本地路径还需启动 Ollama 并拉取 `llama3.2:3b`
3. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格，对比回答风格


In [7]:
# ========== 导入：OpenAI SDK + 笔记本流式展示工具 ==========

# 从 openai 导入 OpenAI 客户端类：同一套 API 可调云端或 Ollama 兼容网关
from openai import OpenAI
# Markdown：渲染；display / update_display：首次展示与流式刷新同一块输出
from IPython.display import Markdown, display, update_display


In [26]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2:3b'


In [20]:
# ========== 客户端 + system prompt：定「怎么讲技术概念」 ==========

# 创建默认 OpenAI 客户端（读环境变量 OPENAI_API_KEY）
openai = OpenAI()
# system prompt 保留英文：这是发给模型的指令，改译会改变回答风格/结构
# 要求按 Definition / Functionality / Example 三件套按需组合，并强调技术准确
system_prompt = """
You are a technical communicator who explains concepts with precision and brevity. Your goal is to give learners exactly what they need to understand a concept—nothing more, nothing less.

For each response, draw from three components:
1. **Definition** — a brief, beginner-friendly explanation of what the concept is
2. **Functionality** — how it works or why it matters
3. **Example** — a concrete illustration (code, analogy, or use case)

Decide which components the question actually requires. Simple terminology questions may only need a definition. "How does X work?" questions usually need definition plus functionality. Questions about applying a concept typically warrant all three. Never pad responses with components that don't add value.

Prioritize technical accuracy above all. Use precise terminology, but define any jargon you introduce. When an example is included, keep it minimal and directly tied to the point being made.
"""


In [21]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文
# 练习建议：换成你自己今天看不懂的一行代码，再分别跑下面 GPT / Llama 两格做对比
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [22]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# stream=True：不要等整段生成完，而是持续返回增量 delta
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages= [
        # system：讲解规范；user：具体问题
        {"role": "system","content":  system_prompt},
        {"role": "user", "content": question}
    ],
    stream=True
)
# response：累加已生成的全文，供每次刷新 Markdown
response = ""
# display_id=True：拿到可更新的句柄，后面用 update_display 原地刷新
display_handle = display(Markdown(""), display_id=True)
# 遍历流式事件：每来一块就拼进 response 并刷新展示
for chunk in stream:
    # delta.content 可能为 None（例如结束块），用 or '' 避免 TypeError
    response += chunk.choices[0].delta.content or ''
    # 用同一 display_id 更新，实现「打字机」Markdown 效果
    update_display(Markdown(response), display_id=display_handle.display_id)


### Definition
This code uses a generator to yield authors from a collection of book dictionaries, filtering out any books that do not have an author field.

### Functionality
- The expression `{book.get("author") for book in books if book.get("author")}` creates a set comprehension. It iterates over each `book` in the `books` list, retrieves the value of the "author" key, and includes it in the set only if an author exists (i.e., it's not `None` or empty).
- The `yield from` statement is used to yield each author from the set one at a time, allowing for lazy evaluation. This means authors are produced on-the-fly as they are requested, rather than all at once.

### Example
Consider a list of book dictionaries:

```python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B"},
    {"title": "Book C", "author": "Author 2"}
]

authors = (yield from {book.get("author") for book in books if book.get("author")})
```
In this example, the result would yield "Author 1" and "Author 2" when queried, skipping any books without an author.

In [29]:
# ========== 路径 B：用本地 Llama（OpenAI 兼容 /v1）流式回答 ==========

# Ollama 的 OpenAI 兼容基址（注意 /v1）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 新建指向本地的客户端；api_key 对本地通常任意非空即可
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# 与 GPT 路径相同的 messages / stream 写法，只是 model 换成 MODEL_LLAMA
stream = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages= [
        {"role": "system","content":  system_prompt},
        {"role": "user", "content": question}
    ],
    stream=True
)
# 同样累加全文并原地刷新 Markdown，便于并排对比两模型风格
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)


**Definition**: `yield from` is a syntax feature in Python that allows a generator function to delegate iteration to another iterable.

**Functionality**: This code uses `yield from` to generate a sequence of author names from a list of books. It yields the values generated by iterating over a filtered subset of books. In this case, it only includes books where an "author" key exists in the book's dictionary.

**Example**: The following is equivalent, but more explicit:
```python
authors = []
for book in books:
    if "author" in book:
        authors.append(book["author"])
yield from authors
```
A simpler analog might be a list comprehension for author names: `[book['author'] for book in books if 'author' in book]`. However, the original code achieves the same result using `yield from`, making it more memory-efficient for large datasets.